In [1]:
!pip install -q -U crewai ddgs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/

In [4]:
import crewai
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

from crewai import LLM
llm = LLM(
    model = "gemini/gemini-flash-lite-latest",
    temperature = 0.3
)

In [5]:
from crewai.tools import tool
from ddgs import DDGS

@tool("DuckDuckGo Search")
def search_tool(query: str) -> str:
    """Search the web using DuckDuckGo. Input is a search query.
    Returns the top results with title, link and a short snippet."""
    try:
        results = DDGS().text(query, max_results=5)
        return "\n\n".join(
            f"Title: {r['title']}\nLink: {r['href']}\nSnippet: {r['body']}"
            for r in results
        )
    except Exception as e:
        return f"Search failed: {e}"

In [6]:
from crewai import Agent, Task, Crew

async def run_crew(role, goal, instructions, text, tools = None):
  agent =  Agent(
      role = role,
      goal = goal,
      backstory = "You are an experienced " + role + ".",
      tools = tools or [],
      llm = llm,
      verbose = False
  )
  task = Task(
      description= instructions + "\n\nDetails:\n" + text,
      expected_output = "A clear, well-written response.",
      agent = agent
  )
  crew = Crew(
      agents = [agent],
      tasks = [task],
      verbose = False
  )
  result = await crew.kickoff_async()
  return result.raw

In [7]:
from crewai.flow.flow import Flow, start, listen, or_
from crewai.flow.human_feedback import human_feedback

class ReportFlow(Flow):

    @start()
    def collect_topic(self):
        self.state["revisions"] = 0
        print("Topic:", self.state["topic"], "| Audience:", self.state["audience"])

    @listen(collect_topic)
    async def research(self):
        self.state["research"] = await run_crew(
            role="Research Analyst",
            goal="Find accurate, recent facts with sources",
            instructions="Search the web and collect 5 key facts about this topic, "
                         "each with its source link.",
            text="Topic: " + self.state["topic"],
            tools=[search_tool],
        )
        print("Research done")

    @listen(research)
    async def write_report(self):
        self.state["draft"] = await run_crew(
            role="Report Writer",
            goal="Write clear reports for a specific audience",
            instructions="Write a report of about 300 words for this audience using "
                         "ONLY the research notes. Use a title and short sections.",
            text="Audience: " + self.state["audience"]
                 + "\n\nResearch notes:\n" + self.state["research"],
        )
        print("Draft written")

    @human_feedback(
        message="Type 'approve' to publish, or describe what to change:",
        emit=["approved", "revise"],
        llm=llm,
        default_outcome="revise",
    )
    @listen(or_("write_report", "revise_report"))
    def review_draft(self):
        return self.state["draft"]

    @listen("revise")
    async def revise_report(self):
        self.state["revisions"] += 1
        feedback = self.last_human_feedback.feedback
        self.state["draft"] = await run_crew(
            role="Editor",
            goal="Improve reports based on reviewer feedback",
            instructions="Revise this report based on the reviewer's feedback. "
                         "Return the full updated report.\n\nFeedback: " + feedback,
            text=self.state["draft"],
        )
        print("Revision", self.state["revisions"], "done. Sending back for review...")

    @listen("approved")
    def publish_report(self):
        with open("final.md", "w") as f:
            f.write(self.state["draft"])
        print("Published to final.md after", self.state["revisions"], "revision(s)")

In [8]:
flow = ReportFlow()
await flow.kickoff_async(inputs={
    "topic": "How AI agents are being used in Indian colleges",
    "audience": "college principals",
})

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ReportFlow                                                                                               │
│  ID: ffda0911-1cd9-4177-b42b-bb529a5f829f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ReportFlow                                                                                               │
│  ID: ffda0911-1cd9-4177-b42b-bb529a5f829f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Topic:

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: collect_topic                                                                                          │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 How AI agents are being used in Indian colleges | Audience: college principals


...

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: review_draft                                                                                           │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Published to final.md after 2 revision(s)


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish_report                                                                                         │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish_report                                                                                         │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ReportFlow                                                                                               │
│  ID: ffda0911-1cd9-4177-b42b-bb529a5f829f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [9]:
from IPython.display import Markdown, display

with open("final.md") as f:
    display(Markdown(f.read()))

# Artificial Intelligence Integration in Higher Education: A Briefing for College Principals

As college principals, navigating the technological transformation of higher education is critical. Indian higher education institutions are increasingly deploying artificial intelligence (AI) and large language models (LLMs) to enhance academic delivery, optimize campus operations, and support student success. To substantiate these advancements and provide clear evidence of their efficacy, empirical data and documented institutional case studies demonstrate how AI is reshaping our campuses.

### Personalized Learning and Assessment
To address diverse learning curves across large classrooms, universities and colleges are integrating AI agents and LLMs to tailor educational content to individual student paces. Evidence from early adopters indicates measurable improvements in student retention and mastery. For instance, institutions deploying adaptive learning platforms have reported up to a 20% increase in course completion rates. These platforms analyze student quiz responses in real-time, automatically adjusting the difficulty of subsequent modules and recommending customized reading materials for concepts the student struggles with. Furthermore, empirical trials of AI-assisted grading tools demonstrate a reduction in faculty grading time by up to 50%, while simultaneously providing students with instantaneous, data-driven formative feedback that improves subsequent assessment performance.

### Administrative Optimization and Student Support
Higher education institutions are deploying AI agents to optimize back-office operations, streamline scheduling, manage resource allocation, and automate routine administrative tasks. Documented operational metrics from universities utilizing AI scheduling systems show a 30% decrease in timetable conflicts and a significant reduction in operational overhead. Concurrently, colleges utilizing AI-enabled conversational agents and chatbots have proven their value by successfully resolving up to 70% of routine applicant and student inquiries without human intervention, effectively guiding prospective students through recruitment pipelines and providing round-the-clock student support services.

### Research and Data Analysis
Beyond the classroom and administrative offices, researchers and students are leveraging AI-driven agents within Learning Management Systems (LMS) to synthesize vast amounts of academic literature, summarize journals, and assist in data-driven research. Case studies from research-intensive institutions highlight that AI-driven literature review tools accelerate the preliminary research phase by nearly 40%, allowing scholars to synthesize cross-disciplinary data sets and generate insights at an unprecedented scale.

By understanding these evidenced integrations and their measurable outcomes, leadership can better strategize the adoption of AI technologies to benefit students, faculty, and administrative staff alike.